# Titanic: Boruta Then VIF

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "pyproject.toml").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from catboost_utility.boruta_catboost import BorutaCatBoost
from catboost_utility.vif_catboost import CatBoostVIF

EXAMPLES_ROOT = PROJECT_ROOT / "examples"
CATBOOST_EXPLORATION_PARAMS = {"iterations": 50, "depth": 4, "learning_rate": 0.1}

In [2]:
data_path = EXAMPLES_ROOT / "titanic_data" / "titanic.csv"
df = pd.read_csv(data_path)

print(f"Loaded {data_path.name} with shape {df.shape}")
display(df.head())
display(df.dtypes.rename("dtype").to_frame())

Loaded titanic.csv with shape (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


,dtype
PassengerId,int64
Survived,int64
Pclass,int64
Name,str
Sex,str
Age,float64
SibSp,int64
Parch,int64
Ticket,str
Fare,float64


In [3]:
target = "Survived"
drop_columns = ["PassengerId", "Name", "Ticket", "Cabin"]
X = df.drop(columns=[target, *drop_columns]).copy()
y = df[target].astype(int)

for col in X.select_dtypes(include=["object", "category", "bool"]).columns:
    X[col] = X[col].fillna("missing")

cat_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
print("Dropped PassengerId, Name, Ticket, and Cabin to avoid identifiers and sparse/high-cardinality text fields.")
print(f"Prepared X with shape {X.shape} and target '{target}'")
print("Categorical features:", cat_features)

Dropped PassengerId, Name, Ticket, and Cabin to avoid identifiers and sparse/high-cardinality text fields.
Prepared X with shape (891, 7) and target 'Survived'
Categorical features: ['Sex', 'Embarked']


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_22960\832481927.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include=["object", "category", "bool"]).columns:
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_22960\832481927.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata

In [4]:
boruta = BorutaCatBoost(
    cat_features=cat_features,
    max_iter=15,
    patience=3,
    correction_method="bonferroni",
    task_type="classification",
    catboost_params=CATBOOST_EXPLORATION_PARAMS,
    random_state=42,
)
boruta.fit(X, y)

boruta_selected = boruta.get_feature_names_out()
boruta_result = boruta.get_selection_result()
decision_log = boruta.decision_log_.copy()
if decision_log.empty:
    latest_boruta_decisions = decision_log
else:
    status_order = pd.CategoricalDtype(["confirmed", "tentative", "rejected"], ordered=True)
    latest_boruta_decisions = (
        decision_log.assign(status=decision_log["status"].astype(status_order))
        .sort_values(["feature", "iteration"])
        .groupby("feature", group_keys=False)
        .tail(1)
        .sort_values(["status", "feature"])
        .reset_index(drop=True)
    )

print("Boruta selected features:", boruta_selected)
print("Boruta rejected features:", boruta_result.rejected_features)
print("Boruta tentative features:", boruta_result.tentative_features)
display(latest_boruta_decisions)

if not boruta_selected:
    raise RuntimeError(
        "Boruta did not confirm any features. Increase max_iter or CatBoost iterations and rerun."
    )

Boruta selected features: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
Boruta rejected features: ['Embarked']
Boruta tentative features: []


,iteration,iteration_seed,feature,shadow_max,hits,p_upper,p_lower,adj_p_upper,adj_p_lower,status
0,10,52,Age,1.988667,9,0.010742,0.999023,0.042969,1.000000,confirmed
1,8,50,Fare,2.263620,8,0.003906,1.000000,0.027344,1.000000,confirmed
2,12,54,Parch,2.353288,10,0.019287,0.996826,0.038574,1.000000,confirmed
3,8,50,Pclass,2.263620,8,0.003906,1.000000,0.027344,1.000000,confirmed
4,8,50,Sex,2.263620,8,0.003906,1.000000,0.027344,1.000000,confirmed
5,13,55,SibSp,1.759541,10,0.046143,0.988770,0.046143,0.988770,confirmed
6,10,52,Embarked,1.988667,1,0.999023,0.010742,1.000000,0.042969,rejected


In [5]:
X_boruta = X[boruta_selected].copy()
vif_cat_features = X_boruta.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

vif = CatBoostVIF(
    cat_features=vif_cat_features,
    threshold=5.0,
    scoring_method="holdout",
    holdout_fraction=0.2,
    n_jobs=1,
    catboost_params=CATBOOST_EXPLORATION_PARAMS,
    random_state=42,
)
vif_result = vif.fit_eliminate(X_boruta)

print("VIF-retained features:", vif_result.selected_features)
print("VIF-dropped features:", vif_result.rejected_features)
display(vif_result.metrics)

elimination_history = pd.DataFrame(vif_result.config["elimination_history"])
if elimination_history.empty:
    print("No VIF eliminations were needed at the current threshold.")
else:
    display(elimination_history)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_22960\1810042448.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  vif_cat_features = X_boruta.select_dtypes(include=["object", "category", "bool"]).columns.tolist()


VIF-retained features: ['Parch', 'SibSp', 'Age', 'Fare', 'Sex']
VIF-dropped features: ['Pclass']


,feature,vif,r_squared,is_categorical,clamped
0,Parch,1.562867,0.360150,False,False
1,SibSp,1.435937,0.303591,False,False
2,Age,1.423596,0.297554,False,False
3,Fare,1.184836,0.156001,False,False
4,Sex,1.046809,0.044716,True,False


,iteration,dropped_feature,dropped_vif,remaining_features
0,0,Pclass,7.141382,5


In [6]:
final_features = vif_result.selected_features
X_final = X_boruta[final_features].copy()

print("Final feature set:", final_features)
display(X_final.head())

Final feature set: ['Parch', 'SibSp', 'Age', 'Fare', 'Sex']


,Parch,SibSp,Age,Fare,Sex
0,0,1,22.0,7.2500,male
1,0,1,38.0,71.2833,female
2,0,0,26.0,7.9250,female
3,0,1,35.0,53.1000,female
4,0,0,35.0,8.0500,male
